# Week 81, Day 2: Caching

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w081d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from functools import lru_cache

calls = []

@lru_cache(maxsize=2)
def ticket(n):
    calls.append(n)
    return f"ticket {n}"

for n in [1, 2, 1, 3, 1, 2]:
    ticket(n)
print(calls)
print(ticket.cache_info().hits, ticket.cache_info().misses)

In [ ]:
def average_ms(hit_rate, cache_ms, source_ms):
    return hit_rate * cache_ms + (1 - hit_rate) * (cache_ms + source_ms)

for hit_rate in (0.0, 0.5, 0.9, 0.99):
    print(f"hit rate {hit_rate:.0%}: {average_ms(hit_rate, 1, 20):.1f} ms, database sees {1 - hit_rate:.0%} of reads")

In [ ]:
db = {7: "open"}
cache = {}
reads = []

def get_status(ticket_id):
    if ticket_id not in cache:
        reads.append(ticket_id)
        cache[ticket_id] = db[ticket_id]
    return cache[ticket_id]

def set_status(ticket_id, status):
    db[ticket_id] = status
    cache.pop(ticket_id, None)        # invalidate: the next read reloads

print(get_status(7), get_status(7))
set_status(7, "resolved")
print(get_status(7), reads)

> **Key idea.** Caches keep copies close to the reader: browser, CDN, reverse proxy, app memory, or a shared cache like Redis. Hit rate decides the payoff (average = hit × cache + miss × (cache + source)); cache data that's read often, changes rarely and is costly. Evict with LRU (OrderedDict, move_to_end, popitem(last=False)), LFU or FIFO; expire with TTLs read from an injected clock; cache "not found" briefly; add jitter. Cache-aside reads load on a miss, and writes update the database then delete the entry; read-through, write-through and write-behind trade freshness, speed and safety; TTLs are the safety net. Stop stampedes with request coalescing or stale-while-revalidate (serve stale, refresh once). Keys must include everything the answer depends on (user or role, normalised path, sorted parameters); personal responses never go into shared caches.

### Exercise: An LRU cache

Write a class `LRUCache(capacity)` with `get(key, default=None)` (a hit marks the key as most recently used), `put(key, value)` (adding or replacing a value makes it most recently used; if the cache is then over capacity, evict the least recently used key and add it to `self.evicted`), `__len__`, `__contains__` (which does **not** count as a use) and `stats()`, returning `{"hits": H, "misses": M, "hit_rate": RATE}` with the rate rounded to 2 places (`0.0` before any lookup). Use `collections.OrderedDict`.

In [ ]:
from collections import OrderedDict

class LRUCache:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lru():\n    \"recency decides\"\n    c = LRUCache(2)\n    c.put(\"a\", 1)\n    c.put(\"b\", 2)\n    assert c.get(\"a\") == 1\n    c.put(\"c\", 3)\n    assert \"b\" not in c and c.evicted == [\"b\"] and len(c) == 2\n    assert c.get(\"b\", \"missing\") == \"missing\"\n    c.put(\"a\", 10)\n    c.put(\"d\", 4)\n    assert c.evicted == [\"b\", \"c\"] and c.get(\"a\") == 10 and \"d\" in c\n    assert c.stats() == {\"hits\": 2, \"misses\": 1, \"hit_rate\": 0.67}\n    assert LRUCache(3).stats() == {\"hits\": 0, \"misses\": 0, \"hit_rate\": 0.0}\n\ndef test_contains_is_not_a_use():\n    \"peeking doesn't refresh\"\n    c = LRUCache(2)\n    c.put(\"a\", 1)\n    c.put(\"b\", 2)\n    assert \"a\" in c\n    c.put(\"c\", 3)\n    assert c.evicted == [\"a\"]")

### Exercise: Entries that expire

Write a class `TTLCache(clock, ttl, negative_ttl=None)`. `set(key, value)` stores the value with an expiry time of `clock() + ttl`; but if the value is `None` (a "not found" answer), it's stored with `negative_ttl` instead, or not stored at all if `negative_ttl` is `None`. `get(key)` returns `(True, value)` for an entry that hasn't expired (it's expired when `clock()` is **at or after** its expiry; expired entries are removed), or `(False, None)`. `delete(key)` removes an entry if present. `purge()` removes every expired entry and returns how many it removed.

In [ ]:
class TTLCache:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ttl():\n    \"fresh, then gone\"\n    class Clock:\n        t = 100.0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    c = TTLCache(clock, 60, negative_ttl=5)\n    c.set(\"ticket:7\", {\"status\": \"open\"})\n    c.set(\"ticket:99\", None)\n    assert c.get(\"ticket:7\") == (True, {\"status\": \"open\"}) and c.get(\"ticket:99\") == (True, None)\n    clock.t = 105\n    assert c.get(\"ticket:99\") == (False, None) and c.get(\"ticket:7\")[0] is True\n    clock.t = 160\n    assert c.get(\"ticket:7\") == (False, None)\n    c.set(\"a\", 1)\n    c.set(\"b\", 2)\n    c.delete(\"a\")\n    c.delete(\"nothing\")\n    assert c.get(\"a\") == (False, None)\n    clock.t = 300\n    c.set(\"c\", 3)\n    assert c.purge() == 1 and c.get(\"c\") == (True, 3)\n    plain = TTLCache(clock, 10)\n    plain.set(\"x\", None)\n    assert plain.get(\"x\") == (False, None) and plain.purge() == 0")

### Exercise: Cache-aside with invalidation

Write a class `CachedTickets(db, cache)`: `db` is a dict standing in for the database (ticket id to a dict), and `cache` is a dict standing in for the cache. Count database reads in `self.db_reads`. `get(ticket_id)`: from the cache if there, else read the database (counting it), store a **copy** of the ticket in the cache, and return it (a missing ticket returns `None` and isn't cached). `update(ticket_id, **changes)`: change the ticket in the database, then delete its cache entry. `delete(ticket_id)`: remove it from the database and the cache. Values returned from `get` must not let a caller change the cached copy: return a copy each time.

In [ ]:
class CachedTickets:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_aside():\n    \"reads load once; writes invalidate\"\n    db = {7: {\"id\": 7, \"status\": \"open\"}, 8: {\"id\": 8, \"status\": \"open\"}}\n    t = CachedTickets(db, {})\n    assert t.get(7) == {\"id\": 7, \"status\": \"open\"} and t.get(7)[\"status\"] == \"open\" and t.db_reads == 1\n    t.update(7, status=\"resolved\")\n    assert t.get(7)[\"status\"] == \"resolved\" and t.db_reads == 2\n    t.get(7)[\"status\"] = \"hacked\"\n    assert t.get(7)[\"status\"] == \"resolved\" and db[7][\"status\"] == \"resolved\"\n    assert t.get(99) is None and t.get(99) is None and t.db_reads == 4\n    t.get(8)\n    t.delete(8)\n    assert t.get(8) is None and 8 not in db")

### Exercise: Stale while revalidate

Write a class `SWRCache(clock, ttl, stale_for)`. `get(key, loader)` returns a value: if the key is cached and younger than `ttl`, return it (a **hit**); if it's older than `ttl` but younger than `ttl + stale_for`, return the stale value at once and queue **one** refresh for that key (in `self.queue`, a list, never holding the same key twice); otherwise (missing, or too old) call `loader(key)` now, store the value with the current time, and return it (a **miss**). `refresh(loaders)` runs the queued refreshes (with the given dict from key to loader function), storing each new value with the current time, empties the queue, and returns the number refreshed. Count `self.hits`, `self.stale` and `self.misses`.

In [ ]:
class SWRCache:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_swr():\n    \"a stampede becomes one refresh\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    calls = []\n    def load(key):\n        calls.append(key)\n        return f\"{key} v{len(calls)}\"\n    c = SWRCache(clock, ttl=60, stale_for=30)\n    assert c.get(\"stats\", load) == \"stats v1\" and calls == [\"stats\"]\n    clock.t = 59\n    assert c.get(\"stats\", load) == \"stats v1\"\n    clock.t = 70\n    for _ in range(1000):\n        assert c.get(\"stats\", load) == \"stats v1\"\n    assert calls == [\"stats\"] and c.queue == [\"stats\"] and c.stale == 1000\n    assert c.refresh({\"stats\": load}) == 1 and c.queue == [] and calls == [\"stats\", \"stats\"]\n    assert c.get(\"stats\", load) == \"stats v2\" and c.hits == 2\n    clock.t = 70 + 91\n    assert c.get(\"stats\", load) == \"stats v3\" and c.misses == 2")

### Exercise: Which eviction policy?

Write `hit_rate(trace, capacity, policy)`: replay a list of keys (`trace`) through a cache holding at most `capacity` keys, where every miss loads the key into the cache (evicting one if it's full), and return the hit rate rounded to 2 places. `policy` is `"lru"` (evict the least recently used), `"fifo"` (evict the one that was loaded earliest; hits don't change anything) or `"lfu"` (evict the one with the fewest hits plus loads so far; ties go to the least recently used). Then `compare(trace, capacity)`: a dict from each policy name to its hit rate.

In [ ]:
def hit_rate(trace, capacity, policy):
    ___

def compare(trace, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_policies():\n    \"different workloads, different winners\"\n    loop = list(\"abcabcabcd\" * 3)\n    assert compare(loop, 3) == {\"lru\": 0.6, \"fifo\": 0.6, \"lfu\": 0.73}\n    recent = list(\"aaabbbcccdddaaa\")\n    assert hit_rate(recent, 2, \"lru\") == 0.67 and hit_rate(recent, 2, \"fifo\") == 0.67\n    hot = [\"x\", \"a\", \"x\", \"b\", \"x\", \"c\", \"x\", \"d\", \"x\", \"e\", \"x\", \"a\"]\n    assert compare(hot, 2) == {\"lru\": 0.42, \"fifo\": 0.25, \"lfu\": 0.42}\n    assert hit_rate([], 2, \"lru\") == 0.0")

### Exercise: Keys that can't leak

Write `cache_key(method, path, query, user_id=None, role=None, personal=False)`: a string key for a response. Only `GET` and `HEAD` are cacheable: for anything else return `None`. The path is lower-cased with any trailing `/` removed (except for `/` itself). The query (a dict, possibly with list values for repeated parameters) becomes `name=value` pairs sorted by name then value, joined with `&`, dropping any parameter named `token` or `_` (cache-busters). The key is `METHOD PATH?QUERY` (without `?` if the query is empty), then `|role=ROLE` if a role is given, then `|user=ID` if `personal` is true; a personal response with no user id can't be cached safely, so raise `ValueError("personal responses need a user id")`.

In [ ]:
def cache_key(method, path, query, user_id=None, role=None, personal=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_keys():\n    \"everything the answer depends on, nothing else\"\n    assert cache_key(\"GET\", \"/Tickets/\", {\"status\": \"open\", \"page\": \"2\"}) == \"GET /tickets?page=2&status=open\"\n    assert cache_key(\"GET\", \"/tickets\", {\"page\": \"2\", \"status\": \"open\"}) == cache_key(\"GET\", \"/tickets/\", {\"status\": \"open\", \"page\": \"2\"})\n    assert cache_key(\"GET\", \"/tickets\", {\"tag\": [\"b\", \"a\"], \"_\": \"1699\", \"token\": \"abc\"}) == \"GET /tickets?tag=a&tag=b\"\n    assert cache_key(\"GET\", \"/\", {}) == \"GET /\" and cache_key(\"POST\", \"/tickets\", {}) is None\n    assert cache_key(\"GET\", \"/stats\", {}, role=\"agent\") == \"GET /stats|role=agent\"\n    assert cache_key(\"HEAD\", \"/me\", {}, user_id=7, personal=True) == \"HEAD /me|user=7\"\n    assert cache_key(\"GET\", \"/me\", {}, user_id=7, personal=True) != cache_key(\"GET\", \"/me\", {}, user_id=8, personal=True)\n    try:\n        cache_key(\"GET\", \"/me\", {}, personal=True)\n    except ValueError as e:\n        assert str(e) == \"personal responses need a user id\"\n    else:\n        raise AssertionError(\"personal without a user\")")

### Bug hunt: Someone else's tickets

The response cache keys on the path alone, so after Ada loads her ticket list, Bob is served **Ada's**: this prints `bob sees: ['Ada: printer']`. Include the user in the key, so it prints `bob sees: ['Bob: login']`.

In [ ]:
TICKETS = {"ada": ["Ada: printer"], "bob": ["Bob: login"]}
cache = {}

def list_tickets(user, path):
    key = path
    if key not in cache:
        cache[key] = TICKETS[user]
    return cache[key]

list_tickets("ada", "/tickets")
print("bob sees:", list_tickets("bob", "/tickets"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the user is part of the key\"\n    expect_output(\"bob sees: ['Bob: login']\")")

### Bug hunt: A status that never changed

The update writes the database but forgets the cache, so readers keep seeing the old status and this prints `after update: open`. Invalidate the entry when the ticket changes, so it prints `after update: resolved`.

In [ ]:
db = {7: "open"}
cache = {}

def get(ticket_id):
    if ticket_id not in cache:
        cache[ticket_id] = db[ticket_id]
    return cache[ticket_id]

def update(ticket_id, status):
    db[ticket_id] = status

get(7)
update(7, "resolved")
print("after update:", get(7))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"delete on write\"\n    expect_output(\"after update: resolved\")")

### Bug hunt: Entries that never expired

The expiry check compares the wrong way round: an entry counts as fresh only **after** its expiry time, so a fresh entry is reloaded and an expired one is served. This prints `at 30: loaded, at 90: cached`. Fix it, so it prints `at 30: cached, at 90: loaded`.

In [ ]:
cache = {"price": (12.50, 60)}      # value, expires at t=60

def lookup(now):
    value, expires = cache["price"]
    return "cached" if now > expires else "loaded"

print(f"at 30: {lookup(30)}, at 90: {lookup(90)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fresh until it expires\"\n    expect_output(\"at 30: cached, at 90: loaded\")")

### Puzzle: A cache-aside read and write

Arrange a cache-aside read and a write that keeps the cache honest. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the average read in ms with a 90% hit rate, a 1 ms cache and a 20 ms database
q = ___   # "lru", "lfu" or "fifo": evicts the entry untouched for longest
r = ___   # on a write, should cache-aside "update" or "delete" the cached entry?
s = ___   # the Cache-Control directive that stops shared caches storing a personal response
t = ___   # the name for many requests reloading one expired entry at once

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3.0 and q == \"lru\" and r == \"delete\", \"p, q and r\"\n    assert s == \"private\" and t in (\"stampede\", \"thundering herd\"), \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a caching decorator

Write a decorator factory `cached(ttl, maxsize, clock)`. The decorated function's results are cached by its arguments (the key is `(args, tuple(sorted(kwargs.items())))`), with LRU eviction beyond `maxsize` entries and expiry `ttl` seconds after an entry was stored (expired when `clock()` is at or after the expiry). Exceptions are **not** cached (they propagate, and the next call tries again). The wrapper keeps the function's name and docstring (`functools.wraps`) and has three attributes: `cache_info()` returning `{"hits": H, "misses": M, "size": S}`, `cache_clear()` emptying the cache (counts too), and `invalidate(*args, **kwargs)` removing the entry for those arguments, returning `True` if there was one.

In [ ]:
import functools
from collections import OrderedDict

def cached(ttl, maxsize, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cached():\n    \"hits, misses, expiry and eviction\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    calls = []\n\n    @cached(ttl=60, maxsize=2, clock=clock)\n    def stats(team, *, detail=False):\n        \"\"\"Ticket statistics for a team.\"\"\"\n        calls.append((team, detail))\n        return f\"{team}:{detail}:{len(calls)}\"\n\n    assert stats.__name__ == \"stats\" and stats.__doc__ == \"Ticket statistics for a team.\"\n    assert stats(\"red\") == \"red:False:1\" and stats(\"red\") == \"red:False:1\"\n    assert stats(\"red\", detail=True) == \"red:True:2\"\n    stats(\"blue\")\n    assert stats.cache_info() == {\"hits\": 1, \"misses\": 3, \"size\": 2}\n    assert stats(\"red\") == \"red:False:4\"\n    clock.t = 59\n    assert stats(\"blue\") == \"blue:False:3\"\n    clock.t = 60\n    assert stats(\"blue\") == \"blue:False:5\"\n    assert stats.invalidate(\"blue\") is True and stats.invalidate(\"blue\") is False\n    stats.cache_clear()\n    assert stats.cache_info() == {\"hits\": 0, \"misses\": 0, \"size\": 0}\n\ndef test_errors_not_cached():\n    \"a failure is retried\"\n    class Clock:\n        def __call__(self):\n            return 0\n    attempts = []\n\n    @cached(ttl=60, maxsize=10, clock=Clock())\n    def flaky(n):\n        attempts.append(n)\n        if len(attempts) == 1:\n            raise ConnectionError(\"database down\")\n        return n * 2\n\n    try:\n        flaky(5)\n    except ConnectionError:\n        pass\n    else:\n        raise AssertionError(\"the error should propagate\")\n    assert flaky(5) == 10 and flaky(5) == 10 and attempts == [5, 5]\n    assert flaky.cache_info() == {\"hits\": 1, \"misses\": 2, \"size\": 1}")

## Warm-up

`functools.lru_cache` (Phase 5) keeps the two most recently **used** answers. Fetching 3 pushed out 2 (used less recently than 1), so 2 had to be loaded again at the end. Which entry to throw out when space runs out, and when an entry is too old to trust, are the two questions every cache has to answer.

## Where caches live

A cache keeps copies of answers somewhere faster than where they came from. Between a user and the database there are several places to put one:

| Where | Example | Shared by |
|---|---|---|
| the browser | `Cache-Control: max-age=60` on a response (Week 76) | one user |
| a CDN (content delivery network) | images, scripts and public pages copied to servers near users | everyone |
| a reverse proxy | a cache in front of the app servers | everyone |
| the app's memory | a dict, `lru_cache` | one app instance |
| a distributed cache | Redis or Memcached, used by every instance | every instance |

The payoff is measured by the **hit rate**: the share of requests answered from the cache. If the database takes 20 ms and the cache 1 ms, a 90% hit rate makes the average read `0.9 × 1 + 0.1 × 21 = 3 ms` (a miss pays for both the cache lookup and the database), and the database sees a tenth of the reads (yesterday's `db_read_replicas` used exactly this).

Caches suit data that is **read often**, **changes rarely**, and is **expensive** to produce. They're a poor fit for data that must always be exactly current (a bank balance during a transfer) or that's rarely read twice.

## Eviction and expiry

**Eviction** decides what to drop when the cache is full:

- **LRU** (least recently used): drop the entry untouched for longest. Recent use predicts future use for most workloads, and it's cheap: an `OrderedDict`, moving an entry to the end on each use (`move_to_end`) and dropping from the front (`popitem(last=False)`).
- **LFU** (least frequently used): drop the entry used least often; good for steady popularity, slow to adapt when it changes.
- **FIFO** (first in, first out): drop the oldest entry, however popular; simple, usually worse.

**Expiry** decides when an entry is too old to trust: each entry gets a **time to live** (TTL), and a lookup after it expires counts as a miss. The TTL is a promise about staleness: "this answer may be up to 60 seconds out of date". Choose it per kind of data: a product description, an hour; a stock level, seconds; a user's permissions, short, or invalidate explicitly.

Two useful variations: **negative caching** stores "not found" answers too (briefly), so repeated requests for something missing don't all reach the database; and **jitter** adds a small random amount to each TTL, so entries written together don't all expire at the same moment.

As always (Weeks 77 and 78), the cache reads the time from an injected **clock**, so tests can step past an expiry instead of sleeping.

## Keeping caches honest

The famous quip: there are only two hard things in computer science, cache invalidation and naming things. When the data changes, the cached copy is **stale**. The strategies:

- **Cache-aside** (the most common): the app reads the cache; on a miss it reads the database and stores the answer. On a **write**, it updates the database and then **deletes** the cached entry, so the next read loads the new value. Deleting is safer than updating the cache, because two writes racing to update the cache can leave the older value there.
- **Read-through**: the cache itself loads missing values (the app only talks to the cache).
- **Write-through**: every write goes to the cache and the database together; reads are always fresh, writes are slower.
- **Write-behind**: writes go to the cache and are copied to the database later; very fast, but a crash can lose writes.

Whatever the strategy, a TTL is the safety net: even if an invalidation is missed, the stale value can't outlive its TTL.

## Stampedes and keys

**Stampedes.** A popular entry expires, and in the next second a thousand requests all miss and all ask the database for the same thing: the very load the cache existed to prevent, arriving all at once (a "thundering herd"). Two cures:

- **Request coalescing** ("single flight"): only the first request for a missing key loads it; the others wait for that one answer.
- **Stale-while-revalidate**: keep an entry a little **past** its TTL; when it's stale but not yet too old, serve the stale value at once and schedule **one** background refresh. Users never wait, and the database sees one request. HTTP has the same idea as a header: `Cache-Control: max-age=60, stale-while-revalidate=30`.

**Keys.** A cache key must include **everything the answer depends on**. Cache `/tickets` under the key `"/tickets"` and the first user's ticket list is served to everyone: a data leak caused by a cache, and a real, recurring kind of incident. So the key includes the user (or their role, if the answer depends only on that), the normalised path, and the query parameters in a fixed order (so `?a=1&b=2` and `?b=2&a=1` share an entry). Responses that are personal should only ever go into per-user caches; shared caches (CDNs, proxies) must be told with `Cache-Control: private` or `no-store`.

### On your own computer

Redis is the usual shared cache (`pip install redis`): `r.set("ticket:7", data, ex=60)` stores with a 60-second TTL, and `r.get("ticket:7")` reads; its `maxmemory-policy allkeys-lru` setting makes it evict like today's LRU. For HTTP, `Cache-Control`, `ETag` and `Vary` headers (Week 76) let browsers and CDNs cache for you.

## Practice

## Quiz

## Daily challenge

## Before you go

Open your browser's developer tools on a site you use, reload, and look at the `Cache-Control` headers on its images, scripts and pages: you'll see long TTLs on files whose names change with each release, and short or none on the pages themselves. Tomorrow: **queues**, for work that doesn't need to happen while the user waits.